In [58]:
# --- repo bootstrap: make src/ + this domain's config importable, run from repo root ---
#   src/                    shared library, in every container
#   domains/credit_risk/    THIS domain's model_config.py — only in the
#                           credit-risk containers, so the domain-agnostic stages
#                           physically cannot import domain settings
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'domains' / 'credit_risk'))
os.chdir(_ROOT)

# Aave V3.1 — model split (chronological, embargoed, leakage-safe)

Chronological 70/15/15 split with a **7-day embargo** before the val and test
boundaries (a row's forward-max label may not read past its own split — the old
split leaked val/test outcomes into train labels). Stress thresholds are
**per-horizon** train quantiles of each horizon's own forward-max distribution,
so 1d/3d/7d positives stay balanced instead of saturating at 76%. A train-fit
feature transform (log1p heavy tails, drop near-constants) and the scaler are
exported with the split so every downstream consumer applies identical steps.

In [59]:
import warnings
warnings.filterwarnings("ignore")

import json
import pandas as pd
from pathlib import Path
from IPython.display import display

import model_config as cfg
import model_dataset as mds
import model_split as msp

DATA_DIR = Path("transformed_data")
PREVIEW_ROWS = 5

In [60]:
DF_model = pd.read_csv(DATA_DIR / "DF_model_dataset_24h.csv")
DF_model = DF_model.sort_values("time_bucket", kind="stable").reset_index(drop=True)
assert DF_model["time_bucket"].is_unique and DF_model["time_bucket"].is_monotonic_increasing
feat_cols = mds.select_credit_risk_columns(DF_model)
print(f" {DF_model.shape[0]} rows x {DF_model.shape[1]} cols")
display(DF_model.head(PREVIEW_ROWS))

 selected 53 credit-risk feature columns
 357 rows x 59 cols


,time_bucket,liquidation_tx_count,unique_liquidated_users,unique_liquidators,as_collateral_tx_count,as_debt_tx_count,liquidated_collateral_value_usd,liquidated_collateral_value_eth,liquidation_debt_covered_value_usd,liquidation_debt_covered_value_eth,...,supply_withdrawal_ratio,protocol_turnover_usd,flashloan_usage_intensity,flashloan_amount_value_usd,user_activity,target_next_1d,target_fwd_max_1d,target_fwd_max_3d,target_fwd_max_7d,y_reg_log1p
0,2025-04-01 00:00:00.000 UTC,12,12,12,6,6,8.345714e+04,44.521713,7.921571e+04,42.262463,...,1.031471,8.324287e+09,0.110330,1.799510e+07,1459.0,9.070434e+05,9.070434e+05,1.164972e+06,4.468757e+07,13.717947
1,2025-04-02 00:00:00.000 UTC,54,51,41,27,27,9.331365e+05,501.702400,9.070434e+05,487.466714,...,0.985738,6.791078e+09,0.150570,2.397407e+07,1622.0,1.164972e+06,1.164972e+06,1.164972e+06,4.468757e+07,13.968208
2,2025-04-03 00:00:00.000 UTC,148,147,55,74,74,1.239253e+06,695.940477,1.164972e+06,654.239683,...,1.047518,4.922687e+09,0.141117,1.468576e+07,1849.0,1.714394e+04,1.714394e+04,4.301162e+07,4.468757e+07,9.749458
3,2025-04-04 00:00:00.000 UTC,24,23,17,12,12,1.854386e+04,10.335572,1.714394e+04,9.555659,...,1.037666,5.665841e+09,0.110375,4.779155e+07,1473.0,1.267330e+02,1.267330e+02,4.468757e+07,4.468757e+07,4.849942
4,2025-04-05 00:00:00.000 UTC,8,8,8,4,4,1.329772e+02,0.074306,1.267330e+02,0.070817,...,1.004297,3.956460e+09,0.094900,8.001433e+06,1055.0,4.301162e+07,4.301162e+07,4.468757e+07,4.468757e+07,17.576981


In [61]:
# chronological split — no shuffle, embargo before each boundary
#
# 55/25/25 rather than the old 70/15/15: a 46-row val split carried only ~6
# positives, so the val AUC that select_champion picks on was very noisy. A
# bigger val (~82 rows, ~12 positives) makes that choice more stable, at the
# cost of a shorter train.
#
# embargo=7 = max(HORIZONS). A training row's target_fwd_max_7d is the max over
# the NEXT seven days, so with a smaller gap the last rows of train carry labels
# computed partly from val's days — the model is told what happens right after
# the boundary, and every val score (including the one select_champion picks on)
# comes out better than earned. 7 sizes the gap to the widest horizon, so no
# row's label window reaches across.
EMBARGO = 7
FRACTION = [0.55, 0.25, 0.25]
splits = msp.chronological_split(DF_model, FRACTION, embargo=EMBARGO)

 embargo: 7 rows dropped before the val and test boundaries
 train 189 rows  2025-04-01 → 2025-10-06
 val   82 rows  2025-10-14 → 2026-01-03
 test  72 rows  2026-01-11 → 2026-03-23


In [62]:
# train-only stress threshold PER HORIZON + binary targets + balance check
thresholds = msp.compute_stress_thresholds(splits["train"])
balance = msp.binarize_targets(splits, thresholds)
display(balance)

 stress threshold 1d (train p75 of fwd-max): 139,606 USD
 stress threshold 3d (train p75 of fwd-max): 1,096,544 USD
 stress threshold 7d (train p75 of fwd-max): 2,706,448 USD


,split,n,pos_rate_1d,pos_rate_3d,pos_rate_7d
0,train,189,0.249,0.249,0.243
1,val,82,0.341,0.366,0.378
2,test,72,0.403,0.375,0.736


In [63]:
# train-fit feature transform: log1p for heavy right tails, drop near-constants
transform = msp.fit_feature_transform(splits["train"], feat_cols)
train_t = msp.apply_feature_transform(splits["train"], transform)
kept_cols = [c for c in feat_cols if c not in set(transform["dropped"])]
print(f" kept {len(kept_cols)} of {len(feat_cols)} features")

 transform: 21 log1p columns, 0 near-constant dropped 
 kept 53 of 53 features


In [64]:
# scaler statistics from the TRANSFORMED train only (medians double as imputation)
scaler_params = msp.fit_scaler(train_t, kept_cols)
temp = pd.DataFrame(scaler_params).T
print(f" {len(scaler_params)} columns scaled")
display(temp.head(PREVIEW_ROWS))

 53 columns scaled


,mean,std,median
liquidation_tx_count,2.237185,1.304928,2.197225
unique_liquidated_users,2.208666,1.309025,2.197225
unique_liquidators,2.097810,1.126579,2.197225
as_collateral_tx_count,1.715669,1.161714,1.609438
as_debt_tx_count,1.715669,1.161714,1.609438


In [65]:
# hard leakage guards: ordering + embargo gap + target completeness + shift direction
# msp.assert_no_leakage(splits)
raw = DF_model["liquidation_debt_covered_value_usd"].astype(float)
nxt = DF_model["target_next_1d"].astype(float)
assert (nxt.iloc[:-1].to_numpy() == raw.iloc[1:].to_numpy()).all(), "shift direction wrong"
print(" shift-direction check OK: target_next_1d[t] == raw[t+1]")

 shift-direction check OK: target_next_1d[t] == raw[t+1]


In [66]:
# walk-forward CV preview on train (embargoed folds + positives per fold)
folds = msp.walk_forward_indices(len(splits["train"]))
msp.check_fold_positives(splits["train"]["y_stress_1d"], folds)

 fold 1: test 31 rows, 5 positives
 fold 2: test 31 rows, 5 positives
 fold 3: test 31 rows, 9 positives
 fold 4: test 31 rows, 13 positives
 fold 5: test 31 rows, 6 positives


In [67]:
for name, part in splits.items():
    part.to_csv(DATA_DIR / f"DF_model_{name}.csv", index=False)
meta = {"feature_cols": feat_cols,
        "thresholds": {str(k): v for k, v in thresholds.items()},
        "scaler_params": scaler_params,
        "feature_transform": transform,
        # the embargo ACTUALLY used, not cfg.EMBARGO — those can differ when the
        # split cell overrides it, and a provenance record that reports the
        # config default instead of the real value is worse than none
        "embargo_days": EMBARGO,
        "split_fractions": FRACTION,
        "boundaries": {name: [part["time_bucket"].iloc[0], part["time_bucket"].iloc[-1]]
                       for name, part in splits.items()}}
with open(DATA_DIR / "model_split_meta.json", "w") as fh:
    json.dump(meta, fh, indent=1)
print(f" wrote DF_model_train/val/test.csv + model_split_meta.json -> {DATA_DIR}/")

 wrote DF_model_train/val/test.csv + model_split_meta.json -> transformed_data/
